In [1]:
from autogen_ext.models.openai import OpenAIChatCompletionClient
import os
from dotenv import load_dotenv

# 加载环境变量
load_dotenv()

model_client = OpenAIChatCompletionClient(
        model="qwen-max",
        api_key=os.getenv("DASHSCOPE_API_KEY"),
        base_url=os.getenv("API_BASE_URL"),
        model_info={
            "vision": True,
            "function_calling": True,
            "json_output": True,
            "family": "unknown",
        },
    )

# 1. 使⽤autoGen通过多agent群聊搜集LLM⽂献实例

In [4]:
! pip install "autogen-ext[docker]"

  Obtaining dependency information for asyncio-atexit>=1.0.1 from https://files.pythonhosted.org/packages/65/10/d6abaefa57a52646651fd0383c056280b0853c0106229ece6bb38cd14463/asyncio_atexit-1.0.1-py3-none-any.whl.metadata
  Obtaining dependency information for docker~=7.0 from https://files.pythonhosted.org/packages/e3/26/57c6fb270950d476074c087527a558ccb6f4436657314bfb6cdf484114c4/docker-7.1.0-py3-none-any.whl.metadata
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 2.1 MB/s eta 0:00:00 0:00:01

[notice] A new release of pip is available: 23.2.1 -> 25.0.1
[notice] To update, run: pip install --upgrade pip


In [2]:
from autogen_agentchat.agents import AssistantAgent, UserProxyAgent, CodeExecutorAgent
from autogen_ext.code_executors.docker import DockerCommandLineCodeExecutor


# 创建一个管理员Proxy
user_proxy = UserProxyAgent(
    name="Admin",
    description="人类管理员. 与规划师互动，讨论计划。计划执行需要得到该管理员的批准。",
)

# 创建一个工程师assistant
engineer = AssistantAgent(
    name="Engineer",
    model_client=model_client,
    system_message="""工程师。您遵循已批准的计划。您编写 python/shell 代码来解决任务。将代码包装在指定脚本类型的代码块中。用户无法修改您的代码。因此，不要建议需要其他人修改的不完整代码。如果代码块不打算由执行器执行，请不要使用它。
    不要在一个响应中包含多个代码块。不要要求其他人复制和粘贴结果。检查执行器返回的执行结果。
    如果结果表明存在错误，请修复错误并再次输出代码。建议使用完整代码而不是部分代码或代码更改。如果错误无法修复，或者即使代码成功执行后任务仍未解决，请分析问题，重新审视您的假设，收集您需要的其他信息，并考虑尝试其他方法。
    """,    
)

# 创建一个科学家assistant
scientist = AssistantAgent(
    name="Scientist",
    model_client=model_client,
    system_message="""科学家。你遵循一个已批准的计划。你可以在看到论文摘要后对论文进行分类。你不写代码。""",
)

# 创建一个计划员assistant
planner = AssistantAgent(
    name="Planner",
    system_message="""规划师。提出计划。根据管理员和评论员的反馈修改计划，直到管理员批准。
    该计划可能涉及会写代码的工程师和不会写代码的科学家。
    首先解释一下计划。明确哪一步由工程师执行，哪一步由科学家执行。
    """,
    model_client=model_client,
)

code_executor = DockerCommandLineCodeExecutor(work_dir="../paper")
await code_executor.start()
# 创建一个执行者Proxy
executor = CodeExecutorAgent(
    name="Executor",
    description="执行器。执行工程师编写的代码并报告结果。",
    code_executor = code_executor
)

# 创建一个评论家assistant
critic = AssistantAgent(
    name="Critic",
    system_message="批评家。仔细检查其他代理商的计划、声明和代码并提供反馈。检查计划是否包括添加可验证信息，例如源 URL。",
    model_client=model_client,
)

In [3]:
from autogen_agentchat.teams import RoundRobinGroupChat

# 组合Agent确保每个任务完成
agent_team = RoundRobinGroupChat(
    [user_proxy, engineer, scientist, planner, executor, critic]
)

In [10]:
from autogen_agentchat.ui import Console

stream = agent_team.run_stream(task="从arxiv上找到最新的5篇关于大语言模型的论文，保存到markdown表。")
await Console(stream)

---------- user ----------
从arxiv上找到最新的5篇关于大语言模型的论文，保存到markdown表。
---------- Admin ----------

---------- Engineer ----------
为了从 arXiv 上获取最新的 5 篇关于大语言模型的论文，并将这些信息保存到一个 Markdown 表格中，我们可以使用 Python 的 `arxiv` 库来搜索相关论文。如果您的环境中没有安装这个库，请先安装它。

```bash
pip install arxiv
```

接下来，我们将编写一个 Python 脚本来完成这项任务。此脚本会搜索标题或摘要中包含 "large language model" 的最新论文，并生成一个 Markdown 格式的表格。

```python
import arxiv
from datetime import datetime

# 搜索参数
search_query = "ti:large language model OR abs:large language model"
max_results = 5

# 获取论文列表
papers = arxiv.Search(
    query=search_query,
    max_results=max_results,
    sort_by=arxiv.SortCriterion.SubmittedDate
)

# 准备Markdown表格
markdown_table = "| Title | Authors | Published Date | URL |\n"
markdown_table += "|-------|---------|----------------|-----|\n"

for paper in papers.results():
    authors_str = ", ".join([author.name for author in paper.authors])
    pub_date = datetime.strftime(paper.published, "%Y-%m-%d")
    markdown_table += f"| [{paper.title}]({pa

# 2. 使⽤autoGen查看股价变动实例

In [ ]:
from IPython.display import Image, display

import autogen
from autogen.coding import LocalCommandLineCodeExecutor

In [8]:
from autogen_agentchat.conditions import TextMentionTermination, MaxMessageTermination
from autogen_agentchat.agents import AssistantAgent, CodeExecutorAgent
from autogen_ext.code_executors.docker import DockerCommandLineCodeExecutor
from autogen_agentchat.teams import RoundRobinGroupChat
from autogen_agentchat.ui import Console

# create an AssistantAgent named "assistant"
assistant = AssistantAgent(
    name="coder",
    system_message='''你需要编写 python/shell 代码来解决任务。将代码包装在指定脚本类型的代码块中。用户无法修改您的代码。因此，不要建议需要其他人修改的不完整代码。如果代码块不打算由执行器执行，请不要使用它。
    不要在一个响应中包含多个代码块。不要要求其他人复制和粘贴结果。检查执行器返回的执行结果。
    如果结果表明存在错误，请修复错误并再次输出代码。建议使用完整代码而不是部分代码或代码更改。如果错误无法修复，或者即使代码成功执行后任务仍未解决，请分析问题，重新审视您的假设，收集您需要的其他信息，并考虑尝试其他方法。''',
    model_client = model_client 
)


code_executor = DockerCommandLineCodeExecutor(work_dir="coding")
await code_executor.start()
# create a UserProxyAgent instance named "user_proxy"
executor = CodeExecutorAgent(
    name="executor",
    code_executor=code_executor
)

max_msg_termination = MaxMessageTermination(max_messages=10)
text_termination = TextMentionTermination("TERMINATE")
combined_termination = max_msg_termination | text_termination

team = RoundRobinGroupChat(
    [assistant, executor],
    termination_condition=combined_termination
)

stream = team.run_stream(task="今天是几号，获取长安汽车本月至今的股票变化信息，并且用文字对股票变化进行分析")
await Console(stream)


---------- user ----------
今天是几号，获取长安汽车本月至今的股票变化信息，并且用文字对股票变化进行分析
---------- coder ----------
要完成这个任务，我们需要分两步走：首先获取今天的日期，然后获取长安汽车本月至今的股票变化信息。对于第二步，我们将使用`yfinance`库来获取股票数据，这是一个非常流行的Python库，用于从Yahoo Finance获取金融数据。如果你还没有安装`yfinance`，可以通过pip安装它。

下面我将给出一个完整的Python脚本来实现这一功能。请注意，此脚本需要联网以下载最新的股票数据。

```python
import yfinance as yf
from datetime import datetime, timedelta
import pandas as pd

# 获取当前日期
today = datetime.now()
print(f"今天是: {today.strftime('%Y-%m-%d')}")

# 设置股票代码和时间范围
stock_code = '000625.SZ'  # 长安汽车在深圳证券交易所的代码
start_date = today.replace(day=1)  # 本月的第一天
end_date = today  # 今天

# 下载股票数据
data = yf.download(stock_code, start=start_date, end=end_date)

# 分析数据
if not data.empty:
    # 计算开盘价与收盘价的变化
    price_change = data['Close'] - data['Open']
    total_change = data['Close'].iloc[-1] - data['Open'].iloc[0]
    
    # 输出分析结果
    print("本月至今，长安汽车的股价变化如下:")
    print(data[['Open', 'Close']])
    if total_change > 0:
        print(f"总体来看，自{start_date.strftime('%Y-%m-%d')}以来，股价上涨了{total

TaskResult(messages=[TextMessage(source='user', models_usage=None, metadata={}, content='今天是几号，获取长安汽车本月至今的股票变化信息，并且用文字对股票变化进行分析', type='TextMessage'), TextMessage(source='coder', models_usage=RequestUsage(prompt_tokens=179, completion_tokens=909), metadata={}, content='要完成这个任务，我们需要分两步走：首先获取今天的日期，然后获取长安汽车本月至今的股票变化信息。对于第二步，我们将使用`yfinance`库来获取股票数据，这是一个非常流行的Python库，用于从Yahoo Finance获取金融数据。如果你还没有安装`yfinance`，可以通过pip安装它。\n\n下面我将给出一个完整的Python脚本来实现这一功能。请注意，此脚本需要联网以下载最新的股票数据。\n\n```python\nimport yfinance as yf\nfrom datetime import datetime, timedelta\nimport pandas as pd\n\n# 获取当前日期\ntoday = datetime.now()\nprint(f"今天是: {today.strftime(\'%Y-%m-%d\')}")\n\n# 设置股票代码和时间范围\nstock_code = \'000625.SZ\'  # 长安汽车在深圳证券交易所的代码\nstart_date = today.replace(day=1)  # 本月的第一天\nend_date = today  # 今天\n\n# 下载股票数据\ndata = yf.download(stock_code, start=start_date, end=end_date)\n\n# 分析数据\nif not data.empty:\n    # 计算开盘价与收盘价的变化\n    price_change = data[\'Close\'] - data[\'Open\']\n    total_change = data[\'Close\']

# 3. 使⽤autoGen获取数据并绘制可视化图实例

In [3]:
from autogen_agentchat.agents import AssistantAgent, CodeExecutorAgent
from autogen_ext.code_executors.docker import DockerCommandLineCodeExecutor
from autogen_agentchat.teams import RoundRobinGroupChat
from autogen_agentchat.ui import Console

code_executor = DockerCommandLineCodeExecutor(work_dir="groupchat")
await code_executor.start()
executor = CodeExecutorAgent(
    name="Executor",
    code_executor=code_executor
)
coder = AssistantAgent(
    name="Coder",  
    model_client=model_client,
)
critic = AssistantAgent(
    name="Critic",
    system_message="""评论家。您是一位乐于助人的助手，能够通过提供 1（差）- 10（好）的分数来评估给定可视化代码的质量，同时提供明确的理由。您必须为每次评估考虑可视化最佳实践。具体来说，您可以从以下维度仔细评估代码
- 错误（bug）：是否存在错误、逻辑错误、语法错误或拼写错误？代码编译失败的原因是什么？应该如何修复？如果存在任何错误，错误分数必须小于 5。
- 数据转换（transformation）：数据是否针对可视化类型进行了适当的转换？例如，数据集是否根据需要进行了适当的过滤、聚合或分组？如果使用日期字段，是否先将日期字段转换为日期对象等？
- 目标合规性（compliance）：代码与指定可视化目标的匹配程度如何？
- 可视化类型 (type)：考虑最佳实践，可视化类型是否适合数据和意图？是否有更有效地传达见解的可视化类型？如果其他可视化类型更合适，则分数必须小于 5。
- 数据编码 (encoding)：数据是否适合可视化类型编码？
- 美学 (aestics)：可视化的美学是否适合可视化类型和数据？

您必须为上述每个维度提供分数。
{错误：0，转换：0，合规性：0，类型：0，编码：0，美观性：0}
不建议代码。
最后，根据上述批评，建议程序员应采取的具体行动清单，以改进代码。
""",
    model_client=model_client,
)

team = RoundRobinGroupChat(
    [coder, executor, critic],
    max_turns=20,
)

stream = team.run_stream(task="从 https://raw.githubusercontent.com/uwdata/draco/master/data/cars.csv 下载数据并绘制可视化图，告诉我们重量和马力之间的关系。将图保存到文件中。在可视化之前打印数据集中的字段。")
await Console(stream)

---------- user ----------
从 https://raw.githubusercontent.com/uwdata/draco/master/data/cars.csv 下载数据并绘制可视化图，告诉我们重量和马力之间的关系。将图保存到文件中。在可视化之前打印数据集中的字段。
---------- Coder ----------
为了完成这个任务，我将使用Python编程语言和一些常用的库如pandas来处理数据，matplotlib或seaborn来进行可视化。下面是执行步骤的代码。

首先，我们需要安装必要的库（如果还没有安装的话）:
```bash
pip install pandas matplotlib seaborn
```

然后，我们可以编写如下脚本来下载数据、打印字段信息，并创建图表：

```python
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# 下载数据
url = "https://raw.githubusercontent.com/uwdata/draco/master/data/cars.csv"
data = pd.read_csv(url)

# 打印数据集中的所有字段
print("Data fields:")
print(data.columns)

# 绘制重量与马力的关系图
plt.figure(figsize=(10, 6))
sns.scatterplot(x='Weight', y='Horsepower', data=data)
plt.title('Relationship between Weight and Horsepower')
plt.xlabel('Weight (lbs)')
plt.ylabel('Horsepower')
plt.grid(True)
plt.tight_layout()

# 保存图像到文件
plt.savefig('weight_vs_horsepower.png')

# 显示图形
plt.show()
```

这段代码做了以下几件事情：
1. 使用 `pandas` 的 `read_csv` 函数从给定的URL加载CSV格式的数据。
2. 

TaskResult(messages=[TextMessage(source='user', models_usage=None, metadata={}, content='从 https://raw.githubusercontent.com/uwdata/draco/master/data/cars.csv 下载数据并绘制可视化图，告诉我们重量和马力之间的关系。将图保存到文件中。在可视化之前打印数据集中的字段。', type='TextMessage'), TextMessage(source='Coder', models_usage=RequestUsage(prompt_tokens=84, completion_tokens=445), metadata={}, content='为了完成这个任务，我将使用Python编程语言和一些常用的库如pandas来处理数据，matplotlib或seaborn来进行可视化。下面是执行步骤的代码。\n\n首先，我们需要安装必要的库（如果还没有安装的话）:\n```bash\npip install pandas matplotlib seaborn\n```\n\n然后，我们可以编写如下脚本来下载数据、打印字段信息，并创建图表：\n\n```python\nimport pandas as pd\nimport matplotlib.pyplot as plt\nimport seaborn as sns\n\n# 下载数据\nurl = "https://raw.githubusercontent.com/uwdata/draco/master/data/cars.csv"\ndata = pd.read_csv(url)\n\n# 打印数据集中的所有字段\nprint("Data fields:")\nprint(data.columns)\n\n# 绘制重量与马力的关系图\nplt.figure(figsize=(10, 6))\nsns.scatterplot(x=\'Weight\', y=\'Horsepower\', data=data)\nplt.title(\'Relationship between Weight and Horsepower\')\nplt.xlabel(\'Weight 

# 4. 使⽤autoGen数据分析实例

In [5]:
from autogen_agentchat.agents import AssistantAgent, CodeExecutorAgent
from autogen_ext.code_executors.docker import DockerCommandLineCodeExecutor
from autogen_agentchat.teams import RoundRobinGroupChat
from autogen_agentchat.ui import Console

coder = AssistantAgent(
    name="Coder",  
    model_client=model_client,
)

code_executor = DockerCommandLineCodeExecutor(work_dir="groupchat2")
await code_executor.start()
executor = CodeExecutorAgent(
    name="Executor",
    code_executor=code_executor
)

critic = AssistantAgent(
    name="Critic",
    system_message="""评论家。您是一位乐于助人的助手，能够通过提供 1（差）- 10（好）的分数来评估给定可视化代码的质量，同时提供明确的理由。您必须为每次评估考虑可视化最佳实践。具体来说，您可以从以下维度仔细评估代码
- 错误（bug）：是否存在错误、逻辑错误、语法错误或拼写错误？代码编译失败的原因是什么？应该如何修复？如果存在任何错误，错误分数必须小于 5。
- 目标合规性（合规性）：代码与指定可视化目标的符合程度如何？

您必须为上述每个维度提供分数。
{bug：0，合规性：0}
不建议代码。
最后，根据上述批评，建议程序员应采取的具体行动清单，以改进代码。
""",
    model_client=model_client,
)

team = RoundRobinGroupChat([coder, executor, critic], max_turns=20)
stream = team.run_stream(task="从 https://raw.githubusercontent.com/vega/vega/main/docs/data/seattle-weather.csv 下载数据并告诉我每种天气的数量。将图保存到文件中。在可视化数据集之前打印数据集中的字段。接受评论者的反馈以改进代码。")
await Console(stream)

---------- user ----------
从 https://raw.githubusercontent.com/vega/vega/main/docs/data/seattle-weather.csv 下载数据并告诉我每种天气的数量。将图保存到文件中。在可视化数据集之前打印数据集中的字段。接受评论者的反馈以改进代码。
---------- Coder ----------
首先，我们需要下载数据并加载它。然后我们将计算每种天气的数量，并将结果可视化。最后，我们将保存图像文件。我将使用Python和pandas库来处理数据，matplotlib进行可视化。

步骤如下：
1. 使用`pandas`读取CSV文件。
2. 打印数据集的前几行以查看字段。
3. 计算每种天气类型的数量。
4. 使用`matplotlib`创建条形图显示每种天气类型的数据。
5. 保存图表到本地文件。

下面是完成这些任务所需的代码：

```python
import pandas as pd
import matplotlib.pyplot as plt

# 下载数据
url = "https://raw.githubusercontent.com/vega/vega/main/docs/data/seattle-weather.csv"
data = pd.read_csv(url)

# 打印数据集的信息
print("Data fields:")
print(data.info())
print("\nFirst few rows of the dataset:")
print(data.head())

# 计算每种天气类型的数量
weather_counts = data['weather'].value_counts()

# 可视化
plt.figure(figsize=(10,6))
weather_counts.plot(kind='bar', color='skyblue')
plt.title('Weather Types in Seattle Dataset')
plt.xlabel('Weather Type')
plt.ylabel('Count')
plt.xticks(rotation=45)  # 旋转x轴标签以便更好地阅读
plt.ti

Error processing publish message for Executor_85e058dd-d36a-4b25-bbee-5c5e4fc781c7/85e058dd-d36a-4b25-bbee-5c5e4fc781c7
Traceback (most recent call last):
  File "/Users/xiaoxia/Documents/LLM/autogen/lib/python3.11/site-packages/autogen_core/_single_threaded_agent_runtime.py", line 510, in _on_message
    return await agent.on_message(
           ^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/xiaoxia/Documents/LLM/autogen/lib/python3.11/site-packages/autogen_core/_base_agent.py", line 113, in on_message
    return await self.on_message_impl(message, ctx)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/xiaoxia/Documents/LLM/autogen/lib/python3.11/site-packages/autogen_agentchat/teams/_group_chat/_sequential_routed_agent.py", line 67, in on_message_impl
    return await super().on_message_impl(message, ctx)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/xiaoxia/Documents/LLM/autogen/lib/python3.11/site-packages/autogen_core/_routed_agent.py", line 485, i

ValueError: Unsupported language: plaintext